In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

C:\Users\kumar_santhosh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\kumar_santhosh\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
reviews = pd.read_csv("Reviews.csv")

reviews = reviews[['Text', 'Score']].dropna()

reviews = reviews.sample(50000, random_state=42)

In [3]:
def label_sentiment(score):
    if score >= 4:
        return 1
    elif score <= 2:
        return 0
    else:
        return 2

reviews['Sentiment'] = reviews['Score'].apply(label_sentiment)

reviews = reviews[reviews['Sentiment'] != 2]

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

reviews['cleaned'] = reviews['Text'].apply(clean_text)

In [5]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(reviews['cleaned'])

sequences = tokenizer.texts_to_sequences(reviews['cleaned'])

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post',
    truncating='post')

y = reviews['Sentiment'].values

In [6]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.4, random_state=42)

In [8]:
model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

C:\Users\kumar_santhosh\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val))

Epoch 1/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 39s 142ms/step - accuracy: 0.8425 - loss: 0.4431 - val_accuracy: 0.8533 - val_loss: 0.4127
Epoch 2/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 37s 127ms/step - accuracy: 0.8527 - loss: 0.4078 - val_accuracy: 0.8552 - val_loss: 0.3986
Epoch 3/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 32s 126ms/step - accuracy: 0.8651 - loss: 0.3633 - val_accuracy: 0.8649 - val_loss: 0.3200
Epoch 4/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 32s 127ms/step - accuracy: 0.9060 - loss: 0.2471 - val_accuracy: 0.9133 - val_loss: 0.2376
Epoch 5/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 33s 131ms/step - accuracy: 0.9394 - loss: 0.1677 - val_accuracy: 0.9156 - val_loss: 0.2218
Epoch 6/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 32s 127ms/step - accuracy: 0.9572 - loss: 0.1211 - val_accuracy: 0.9261 - val_loss: 0.2295
Epoch 7/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 32s 127ms/step - accuracy: 0.9721 - loss: 0.0874 - val_accuracy: 0.9078 - val_loss: 0.2412
Epoch 8/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 33s 129ms/step - accuracy: 0.9819 - loss: 0

In [11]:
loss_lstm, acc_lstm = model_lstm.evaluate(X_test, y_test)
print("LSTM Test Accuracy:", acc_lstm)

174/174 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9071 - loss: 0.3443
LSTM Test Accuracy: 0.9071402549743652


In [12]:
model_cnn_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_cnn_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True)

history_cnn_lstm = model_cnn_lstm.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stop])

Epoch 1/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 26s 89ms/step - accuracy: 0.8441 - loss: 0.4317 - val_accuracy: 0.8819 - val_loss: 0.3169
Epoch 2/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 22s 87ms/step - accuracy: 0.8965 - loss: 0.2592 - val_accuracy: 0.9154 - val_loss: 0.2148
Epoch 3/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 22s 86ms/step - accuracy: 0.9355 - loss: 0.1723 - val_accuracy: 0.9258 - val_loss: 0.1970
Epoch 4/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.9560 - loss: 0.1249 - val_accuracy: 0.9331 - val_loss: 0.1911
Epoch 5/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 22s 87ms/step - accuracy: 0.9745 - loss: 0.0808 - val_accuracy: 0.9250 - val_loss: 0.2133
Epoch 6/10
253/253 ━━━━━━━━━━━━━━━━━━━━ 22s 87ms/step - accuracy: 0.9847 - loss: 0.0520 - val_accuracy: 0.9259 - val_loss: 0.2644


In [14]:
loss_cnn, acc_cnn = model_cnn_lstm.evaluate(X_test, y_test)
print("CNN+LSTM Test Accuracy:", acc_cnn)

174/174 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9176 - loss: 0.2278
CNN+LSTM Test Accuracy: 0.9175982475280762


In [15]:
if acc_cnn > acc_lstm:
    best_model = model_cnn_lstm
    print("Selected Model: CNN + LSTM")
else:
    best_model = model_lstm
    print("Selected Model: LSTM")

Selected Model: CNN + LSTM


In [16]:
df = pd.read_csv("final_dashboard_dataset.csv")

In [17]:
df['Customer_Review'] = reviews['Text'].sample(len(df)).values

new_seq = tokenizer.texts_to_sequences(df['Customer_Review'])

new_pad = pad_sequences(
    new_seq,
    maxlen=max_len,
    padding='post',
    truncating='post')

df['Predicted_Sentiment'] = (best_model.predict(new_pad) > 0.5).astype(int)

70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step


In [18]:
df['Predicted_Sentiment'] = df['Predicted_Sentiment'].map({
    1: "Positive",
    0: "Negative"})

In [19]:
print(pd.crosstab(df['Segment'], df['Predicted_Sentiment']))

Predicted_Sentiment   Negative  Positive
Segment                                 
At Risk Customers           67       426
Budget Buyers               82       474
High Value Customers        67       475
New Customers               84       561


In [20]:
df.to_csv("final_with_sentiment.csv", index=False)

In [21]:
best_model.save("best_sentiment_model.h5")